# Workshop 3: Data Preparation Pipeline

Notebook นี้ออกแบบสำหรับ Session 02 (Stat on Campus) เพื่อฝึกกระบวนการเตรียมข้อมูลก่อนทำ EDA

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/toche7/SlideAIDATADGA/blob/main/slides/workshop-03-data-preparation-pipeline.ipynb)

## Learning Objectives
- ดึงข้อมูล Titanic มาเป็น DataFrame
- ตรวจสอบคุณภาพข้อมูล (missing, duplicate, type, outlier)
- ทำ data cleaning และจัดข้อมูลให้อยู่ในรูปแบบที่พร้อมวิเคราะห์
- สรุปผลการเตรียมข้อมูลเป็น checklist เพื่อใช้ต่อใน EDA

In [1]:
# 1) Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 50)
sns.set_theme(style='whitegrid')

In [ ]:
# 2) Load Titanic dataset (with fallback URL)
fallback_url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'

try:
    df = sns.load_dataset('titanic')
    data_source = 'seaborn.titanic'
except Exception:
    df = pd.read_csv(fallback_url)
    data_source = 'Fallback CSV'

print(f'Loaded from: {data_source}')
print(f'Shape: {df.shape}')
df.head()

Loaded from: Fallback CSV
Shape: (5000, 10)


,_id,year,region,province,area,sex,age_group,value,unit,source
0,1,2533,ทั่วประเทศ,รวม,รวม,รวม,รวม,54548530,คน,สำนักงานสถิติแห่งชาติ
1,2,2533,ทั่วประเทศ,รวม,รวม,ชาย,รวม,27061733,คน,สำนักงานสถิติแห่งชาติ
2,3,2533,ทั่วประเทศ,รวม,รวม,หญิง,รวม,27486797,คน,สำนักงานสถิติแห่งชาติ
3,4,2533,ทั่วประเทศ,รวม,ในเขตเทศบาล,รวม,รวม,10215098,คน,สำนักงานสถิติแห่งชาติ
4,5,2533,ทั่วประเทศ,รวม,ในเขตเทศบาล,ชาย,รวม,4933482,คน,สำนักงานสถิติแห่งชาติ


In [3]:
# 3) Quick profile
print('Columns:', list(df.columns))
display(df.info())
display(df.describe(include='all').T.head(10))

Columns: ['_id', 'year', 'region', 'province', 'area', 'sex', 'age_group', 'value', 'unit', 'source']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 10 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   _id        5000 non-null   int64 
 1   year       5000 non-null   int64 
 2   region     5000 non-null   object
 3   province   5000 non-null   object
 4   area       5000 non-null   object
 5   sex        5000 non-null   object
 6   age_group  5000 non-null   object
 7   value      5000 non-null   int64 
 8   unit       5000 non-null   object
 9   source     5000 non-null   object
dtypes: int64(3), object(7)
memory usage: 390.8+ KB


None

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
_id,5000.0,NaN,NaN,NaN,2500.5,1443.520003,1.0,1250.75,2500.5,3750.25,5000.0
year,5000.0,NaN,NaN,NaN,2543.002,8.167823,2533.0,2533.0,2543.0,2553.0,2553.0
region,5000,3,ภาคกลาง,4187,NaN,NaN,NaN,NaN,NaN,NaN,NaN
province,5000,10,รวม,972,NaN,NaN,NaN,NaN,NaN,NaN,NaN
area,5000,3,รวม,1721,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sex,5000,3,รวม,1667,NaN,NaN,NaN,NaN,NaN,NaN,NaN
age_group,5000,18,รวม,288,NaN,NaN,NaN,NaN,NaN,NaN,NaN
value,5000.0,NaN,NaN,NaN,415844.4764,2426742.822436,0.0,5947.0,17696.5,113869.75,65981659.0
unit,5000,1,คน,5000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
source,5000,1,สำนักงานสถิติแห่งชาติ,5000,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# 4) Data quality check: missing values + duplicates
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
quality_table = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})

print('Top missing columns')
display(quality_table.head(15))

dup_count = df.duplicated().sum()
print(f'Duplicate rows: {dup_count}')

Top missing columns


,missing_count,missing_pct
_id,0,0.0
year,0,0.0
region,0,0.0
province,0,0.0
area,0,0.0
sex,0,0.0
age_group,0,0.0
value,0,0.0
unit,0,0.0
source,0,0.0


Duplicate rows: 0


In [ ]:
# 5) Type conversion and standardization
# Convert numeric-like columns
for col in ['age', 'fare', 'sibsp', 'parch', 'pclass']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Standardize text columns
text_cols = [c for c in ['sex', 'embarked', 'embark_town', 'class', 'who', 'deck', 'alive'] if c in df.columns]
for c in text_cols:
    df[c] = df[c].astype('string').str.strip()

preview_cols = [c for c in ['survived', 'pclass', 'sex', 'age', 'fare', 'embarked'] if c in df.columns]
df[preview_cols].head()

In [ ]:
# 6) Handle missing and duplicates for analysis-ready table
df_clean = df.copy()

# Remove exact duplicate rows
df_clean = df_clean.drop_duplicates()

# Fill missing values with practical rules
if {'age', 'sex', 'pclass'}.issubset(df_clean.columns):
    df_clean['age'] = df_clean.groupby(['sex', 'pclass'])['age'].transform(lambda s: s.fillna(s.median()))

if 'embarked' in df_clean.columns:
    embarked_mode = df_clean['embarked'].mode(dropna=True)
    if not embarked_mode.empty:
        df_clean['embarked'] = df_clean['embarked'].fillna(embarked_mode.iloc[0])

if 'fare' in df_clean.columns:
    df_clean['fare'] = df_clean['fare'].fillna(df_clean['fare'].median())

# Drop columns that are mostly missing or IDs not needed for modeling
drop_cols = [c for c in ['deck', 'embark_town', 'alive', 'class', 'who'] if c in df_clean.columns]
df_clean = df_clean.drop(columns=drop_cols)

print('Original shape:', df.shape)
print('Cleaned shape :', df_clean.shape)
print('Remaining missing values:', int(df_clean.isna().sum().sum()))

In [ ]:
# 7) Outlier inspection on fare using IQR
q1 = df_clean['fare'].quantile(0.25)
q3 = df_clean['fare'].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

outliers = df_clean[(df_clean['fare'] < lower) | (df_clean['fare'] > upper)]
print(f'Outlier rows (IQR rule): {len(outliers):,}')
display(outliers[['survived', 'pclass', 'sex', 'age', 'fare']].head())

In [ ]:
# 8) Quick visualization for quality check
plt.figure(figsize=(10, 4))
sns.histplot(df_clean['fare'], bins=40, kde=True)
plt.title('Distribution of Titanic Fare (Cleaned)')
plt.xlabel('fare')
plt.ylabel('count')
plt.show()

In [ ]:
# 9) Build tidy summary dataset
group_cols = [c for c in ['pclass', 'sex'] if c in df_clean.columns]
tidy_df = (
    df_clean
    .groupby(group_cols, dropna=False, as_index=False)['survived']
    .mean()
    .rename(columns={'survived': 'survival_rate'})
)

print('Tidy shape:', tidy_df.shape)
tidy_df.head()

In [ ]:
# 10) Export cleaned outputs
df_clean.to_csv('workshop3_titanic_clean.csv', index=False)
tidy_df.to_csv('workshop3_titanic_tidy.csv', index=False)
print('Saved: workshop3_titanic_clean.csv')
print('Saved: workshop3_titanic_tidy.csv')

## Reflection Questions
1. หลัง cleaning แล้ว จำนวนข้อมูลลดลงเท่าไร และเพราะอะไร?
2. คอลัมน์ไหนใน Titanic มี missing สูงที่สุด และควรจัดการแบบใด?
3. Outlier ของค่า fare ควรเก็บไว้หรือเอาออก เพราะอะไร?
4. หากจะทำ EDA ต่อ ควรตั้งคำถามอะไรเกี่ยวกับ survival rate?